The Oracle Accelerated Data Science (ADS) SDK is maintained by the Oracle Cloud Infrastructure Data Science service team. It speeds up common data science activities by providing tools that automate and/or simplify common data science tasks, along with providing a data scientist friendly pythonic interface to Oracle Cloud Infrastructure (OCI) services, most notably OCI Data Science, Data Flow, Object Storage, and the Autonomous Database. ADS gives you an interface to manage the lifecycle of machine learning models, from data acquisition to model evaluation, interpretation, and model deployment.

## Verify OCI API Connectivity

In [2]:
import oci
profile="DEFAULT"
config = oci.config.from_file(profile_name=profile)

# ----------------------------------------------------------------------------------------
# Print OCI API Configuration parameters. These are retrieved from OCI CLI/API Config file
# ----------------------------------------------------------------------------------------

print("OCI_TENANCY: "+config["tenancy"])
print("OCI USER: "+config["user"])
print("OCI FINGERPRINT: "+config["fingerprint"])
print("OCI_REGION: "+config.get("region"))
print("OCI_KEY_FILE: "+config.get("key_file"))
print("OCI_KEY_CONTENT: "+str(config.get("key_content"))) #Optional (setup in private pem file)
print("OCI_PASSPHRASE: "+str(config.get("passphrase")))   #Optional

OCI_TENANCY: ocid1.tenancy.oc1..aaaaaaaaz62ru446esycbomaye4aohqqjqx2bdamhfuynfegzm2lgbustlna
OCI USER: ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta
OCI FINGERPRINT: d7:e3:95:7b:04:d0:21:48:a6:a5:47:56:83:7a:e9:62
OCI_REGION: us-ashburn-1
OCI_KEY_FILE: /home/datascience/.oci/oci_api_key.pem
OCI_KEY_CONTENT: None
OCI_PASSPHRASE: None


## Setting up AIDP Client

In [3]:
import base64
import json
import os
import sys

import config
from aidp_client import AIDPClient, AIDPAPIError, poll_until_active, print_response

# ---------------------------------------------------------------------------
# Shared client (all tests use this)
# ---------------------------------------------------------------------------

def make_client() -> AIDPClient:
    return AIDPClient(
        region=config.REGION,
        aidp_instance_id=config.AIDP_INSTANCE_ID,
        profile=config.OCI_PROFILE,
    )


In [12]:
# ---------------------------------------------------------------------------
# Print configuration variables
# ---------------------------------------------------------------------------
print("OCI_PROFILE: "+config.OCI_PROFILE)
print("REGION: "+config.REGION)
print("AIDP_INSTANCE_ID: "+config.AIDP_INSTANCE_ID)
print("COMPARTMENT_ID: "+config.COMPARTMENT_ID)
print("")
# Parameters set in config.py after creation of certain AIDP objects such as workspaces or clusters
print("EXISTING_WORKSPACE_NAME: "+config.EXISTING_WORKSPACE_NAME)
print("WORKSPACE_KEY: "+config.WORKSPACE_KEY)
print("CLUSTER_KEY: "+config.CLUSTER_KEY)
print("")
## For private workspace name and network config
print("PVT_WORKSPACE_NAME: "+config.NEW_WORKSPACE_NAME)
print("VCN_ID: "+config.VCN_ID)
print("SUBNET_ID: "+config.SUBNET_ID)
print("NSG_ID: "+config.NSG_ID) #Optional
print("")
## ADW Catalog details
print("ADW_CATALOG_NAME: "+config.CATALOG_NAME)
print("ADW_WALLET_ZIP: "+config.ADW_WALLET_ZIP)
print("ADW_USERNAME: "+config.ADW_USERNAME)
print("ADW_WALLET_PASSWORD: "+config.ADW_WALLET_PASSWORD)
print("ADW_TNS_ALIAS: "+config.ADW_TNS_ALIAS)
print("")
## Compute cluster config details 
print("CLUSTER_NAME: "+config.CLUSTER_NAME)
print("CLUSTER_KEY: "+config.CLUSTER_KEY)
print("DRIVER_OCPUS: "+str(config.DRIVER_OCPUS))
print("WORKER_OCPUS: "+str(config.WORKER_OCPUS))
print("WORKER_MEMORY_GBS: "+str(config.WORKER_MEMORY_GBS))
print("MIN_WORKERS: "+str(config.MIN_WORKERS))
print("")

OCI_PROFILE: DEFAULT
REGION: us-ashburn-1
AIDP_INSTANCE_ID: ocid1.aidataplatform.oc1.iad.amaaaaaaqvm24naafjrto5h43knplzdzk4mnxeypyghj6vxz6rpvr5ckxo7q
COMPARTMENT_ID: ocid1.compartment.oc1..aaaaaaaardd6zegida3d3v2jtbwmexgfsxvorrfhwyjanqea3r3hwj6v55ma

EXISTING_WORKSPACE_NAME: PilotTest
WORKSPACE_KEY: dae0bef7-354b-448b-95c4-1c7b12658e68
CLUSTER_KEY: 7696453b-2df7-4c89-a64f-ac90d668a357

PVT_WORKSPACE_NAME: PilotTestPvt
VCN_ID: ocid1.vcn.oc1.iad.amaaaaaaqvm24naao47cy37xq46rwepi4pt5k4owybzoqhfwf5fqemfwidja
SUBNET_ID: ocid1.subnet.oc1.iad.aaaaaaaauabpq3ab6awidy6q75p72ut2gjxh5hvrll2uh4akbof2wgln2zka
NSG_ID: 

ADW_CATALOG_NAME: PilotADWDB
ADW_WALLET_ZIP: ./Wallet_pilot.zip
ADW_USERNAME: ADMIN
ADW_WALLET_PASSWORD: Welcome#1234!
ADW_TNS_ALIAS: pilot_low

CLUSTER_NAME: Pilot_api_Cluster1
CLUSTER_KEY: 7696453b-2df7-4c89-a64f-ac90d668a357
DRIVER_OCPUS: 2
WORKER_OCPUS: 2
WORKER_MEMORY_GBS: 32
MIN_WORKERS: 1



In [15]:
# ---------------------------------------------------------------------------
# Test 01 — GET AIDP instance (health check / connectivity)
# ---------------------------------------------------------------------------

def test_01():
    """GET /  — verify connectivity and AIDP instance is ACTIVE."""
    print("\n=== test_01: GET AIDP instance ===")
    client = make_client()
    resp = client.get("")
    #print_response(resp, "GET instance")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    data = resp.json()
    state = data.get("lifecycleState", "?")
    print(f"  Instance state: {state}")
    assert state == "ACTIVE", f"Expected ACTIVE, got {state}"
    print("  PASS")
    return data

In [16]:
test_01()


=== test_01: GET AIDP instance ===
  Instance state: ACTIVE
  PASS


{'id': 'ocid1.aidataplatform.oc1.iad.amaaaaaaqvm24naafjrto5h43knplzdzk4mnxeypyghj6vxz6rpvr5ckxo7q',
 'displayName': 'Pilot',
 'compartmentId': 'ocid1.compartment.oc1..aaaaaaaardd6zegida3d3v2jtbwmexgfsxvorrfhwyjanqea3r3hwj6v55ma',
 'dataLakeType': None,
 'createdBy': 'ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta',
 'timeCreated': '2025-10-13T18:13:22.805Z',
 'timeUpdated': '2026-04-20T20:04:23.633Z',
 'aliasKey': '3kcgxll9o0fnv3jc1n6',
 'webSocketEndpoint': None,
 'lifecycleState': 'ACTIVE',
 'lifecycleDetails': None,
 'freeformTags': {},
 'definedTags': {'Oracle-Tags': {'CreatedBy': 'default/rajib.g.ghosh@oracle.com',
   'CreatedOn': '2025-10-13T18:13:22.187Z'}},
 'systemTags': {}}

In [17]:
# ---------------------------------------------------------------------------
# Test 02 — List workspaces
# ---------------------------------------------------------------------------

def test_02():
    """GET /workspaces  — list all workspaces in the AIDP instance."""
    print("\n=== test_02: List workspaces ===")
    client = make_client()
    resp = client.get("/workspaces")
    #print_response(resp, "GET /workspaces")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    items = resp.json()
    if isinstance(items, dict):
        items = items.get("items", [])
    print(f"  Found {len(items)} workspace(s):")
    for ws in items:
        print(f"    key={ws.get('key')}  name={ws.get('displayName')}  state={ws.get('lifecycleState')}")
    print("  PASS")
    return items

In [18]:
test_02()


=== test_02: List workspaces ===
  Found 5 workspace(s):
    key=1fa39760-9691-447c-acbf-011193473dae  name=PilotTestPvt  state=ACTIVE
    key=dae0bef7-354b-448b-95c4-1c7b12658e68  name=PilotTest  state=ACTIVE
    key=25cca23b-0a96-4df4-a7ab-72f231679515  name=test  state=ACTIVE
    key=76037edd-b91e-4e78-9e2e-d79155aaa85f  name=aidp_workspace  state=ACTIVE
    key=5a0a610f-4c36-4379-a3e9-cf42f67c7378  name=pilot  state=ACTIVE
  PASS


[{'key': '1fa39760-9691-447c-acbf-011193473dae',
  'displayName': 'PilotTestPvt',
  'type': 'USER',
  'description': 'Test workspace: PilotTestPvt',
  'timeCreated': '2026-04-21T13:08:03.989Z',
  'timeUpdated': '2026-04-21T13:08:16.517Z',
  'lifecycleState': 'ACTIVE',
  'lifecycleDetails': 'ACTIVE',
  'freeformTags': None,
  'definedTags': None,
  'systemTags': None,
  'createdBy': 'ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta',
  'createdByName': 'Rajib Ghosh',
  'updatedBy': None,
  'updatedByName': None,
  'defaultCatalogKey': None,
  'properties': None,
  'isPrivateNetworkEnabled': False,
  'aicUserSchemaName': None},
 {'key': 'dae0bef7-354b-448b-95c4-1c7b12658e68',
  'displayName': 'PilotTest',
  'type': 'USER',
  'description': 'Updated description via REST API',
  'timeCreated': '2026-04-20T20:56:23.393Z',
  'timeUpdated': '2026-04-21T13:18:08.270Z',
  'lifecycleState': 'ACTIVE',
  'lifecycleDetails': 'ACTIVE',
  'freeformTags': None,
  'definedTa

In [8]:
# ---------------------------------------------------------------------------
# Test 03 — Find existing workspace by name (PATH B)
# ---------------------------------------------------------------------------

def test_03():
    """GET /workspaces then filter by displayName == EXISTING_WORKSPACE_NAME."""
    print(f"\n=== test_03: Find workspace '{config.EXISTING_WORKSPACE_NAME}' ===")
    client = make_client()
    resp = client.get("/workspaces")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    items = resp.json()
    if isinstance(items, dict):
        items = items.get("items", [])

    match = None
    for ws in items:
        if ws.get("displayName") == config.EXISTING_WORKSPACE_NAME:
            match = ws
            break

    if not match:
        print(f"  FAIL: Workspace '{config.EXISTING_WORKSPACE_NAME}' not found.")
        print(f"  Available workspaces:")
        for ws in items:
            print(f"    {ws.get('displayName')} (key={ws.get('key')})")
        sys.exit(1)

    ws_key = match.get("key") or match.get("id")
    print(f"  Found: key={ws_key}  state={match.get('lifecycleState')}")
    print(f"\n  >>> Copy this to config.py WORKSPACE_KEY: {ws_key}")
    print("  PASS")
    return match

In [16]:
test_03()


=== test_03: Find workspace 'PilotTest' ===
  Found: key=dae0bef7-354b-448b-95c4-1c7b12658e68  state=ACTIVE

  >>> Copy this to config.py WORKSPACE_KEY: dae0bef7-354b-448b-95c4-1c7b12658e68
  PASS


{'key': 'dae0bef7-354b-448b-95c4-1c7b12658e68',
 'displayName': 'PilotTest',
 'type': 'USER',
 'description': 'Updated description via REST API',
 'timeCreated': '2026-04-20T20:56:23.393Z',
 'timeUpdated': '2026-04-20T23:01:51.225Z',
 'lifecycleState': 'ACTIVE',
 'lifecycleDetails': 'ACTIVE',
 'freeformTags': None,
 'definedTags': None,
 'systemTags': None,
 'createdBy': 'ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta',
 'createdByName': 'Rajib Ghosh',
 'updatedBy': 'ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta',
 'updatedByName': 'Rajib Ghosh',
 'defaultCatalogKey': None,
 'properties': None,
 'isPrivateNetworkEnabled': False,
 'aicUserSchemaName': None}

In [11]:
# ---------------------------------------------------------------------------
# Test 04 — Create private workspace (PATH A)
# ---------------------------------------------------------------------------

def test_04():
    """POST /workspaces  — create new private workspace with VCN/subnet/NSG (PATH A)."""
    print(f"\n=== test_04: Create workspace '{config.NEW_WORKSPACE_NAME}' ===")

    assert config.VCN_ID and not config.VCN_ID.endswith("..."), \
        "Set VCN_ID in config.py before running test_04"
    assert config.SUBNET_ID and not config.SUBNET_ID.endswith("..."), \
        "Set SUBNET_ID in config.py before running test_04"
    assert config.COMPARTMENT_ID and not config.COMPARTMENT_ID.endswith("..."), \
        "Set COMPARTMENT_ID in config.py before running test_04"

    client = make_client()
    body = {
        "displayName": config.NEW_WORKSPACE_NAME,
        "description": f"Test workspace: {config.NEW_WORKSPACE_NAME}",
        "compartmentId": config.COMPARTMENT_ID,
        "networkConfiguration": {
            "vcnId": config.VCN_ID,
            "subnetId": config.SUBNET_ID,
            "nsgIds": [config.NSG_ID] if config.NSG_ID else [],
        },
    }

    print(f"  Request body:")
    print(json.dumps(body, indent=2))

    resp = client.post("/workspaces", body)
    #print_response(resp, "POST /workspaces")

    assert resp.status_code in (200, 201, 202), \
        f"Expected 200/201/202, got {resp.status_code}: {resp.text[:300]}"

    data = resp.json() if resp.content else {}
    ws_key = data.get("key") or data.get("id")

    if not ws_key:
        print("  No key in response — polling list to find workspace ...")
        deadline_passed = False
        import time
        end = time.time() + 120
        while time.time() < end:
            time.sleep(10)
            list_resp = client.get("/workspaces")
            if list_resp.ok:
                items = list_resp.json()
                if isinstance(items, dict):
                    items = items.get("items", [])
                for item in items:
                    if item.get("displayName") == config.NEW_WORKSPACE_NAME:
                        ws_key = item.get("key") or item.get("id")
                        if ws_key:
                            break
                if ws_key:
                    break

    assert ws_key, "Could not retrieve workspace key after creation"
    print(f"  Workspace key: {ws_key}")

    ws_data = poll_until_active(client, f"/workspaces/{ws_key}",
                                f"workspace '{config.NEW_WORKSPACE_NAME}'",
                                max_wait=600)
    print(f"\n  >>> Copy this to config.py WORKSPACE_KEY: {ws_key}")
    print("  PASS")
    return {"key": ws_key, "data": ws_data}


In [12]:
test_04()


=== test_04: Create workspace 'PilotTestPvt' ===
  Request body:
{
  "displayName": "PilotTestPvt",
  "description": "Test workspace: PilotTestPvt",
  "compartmentId": "ocid1.compartment.oc1..aaaaaaaardd6zegida3d3v2jtbwmexgfsxvorrfhwyjanqea3r3hwj6v55ma",
  "networkConfiguration": {
    "vcnId": "ocid1.vcn.oc1.iad.amaaaaaaqvm24naao47cy37xq46rwepi4pt5k4owybzoqhfwf5fqemfwidja",
    "subnetId": "ocid1.subnet.oc1.iad.aaaaaaaauabpq3ab6awidy6q75p72ut2gjxh5hvrll2uh4akbof2wgln2zka",
    "nsgIds": []
  }
}
  HTTP 201 [POST /workspaces]
{
  "key": "1fa39760-9691-447c-acbf-011193473dae",
  "displayName": "PilotTestPvt",
  "description": "Test workspace: PilotTestPvt",
  "type": "USER",
  "timeCreated": "2026-04-21T13:08:03.989Z",
  "timeUpdated": "2026-04-21T13:08:03.989Z",
  "lifecycleState": "CREATING",
  "lifecycleDetails": null,
  "freeformTags": null,
  "definedTags": null,
  "systemTags": null,
  "createdBy": "ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta",
  

{'key': '1fa39760-9691-447c-acbf-011193473dae',
 'data': {'key': '1fa39760-9691-447c-acbf-011193473dae',
  'displayName': 'PilotTestPvt',
  'description': 'Test workspace: PilotTestPvt',
  'type': 'USER',
  'timeCreated': '2026-04-21T13:08:03.989Z',
  'timeUpdated': '2026-04-21T13:08:16.517Z',
  'lifecycleState': 'ACTIVE',
  'lifecycleDetails': 'ACTIVE',
  'freeformTags': None,
  'definedTags': None,
  'systemTags': None,
  'createdBy': 'ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta',
  'createdByName': 'Rajib Ghosh',
  'updatedBy': None,
  'updatedByName': None,
  'defaultCatalogKey': None,
  'properties': None,
  'isPrivateNetworkEnabled': False,
  'networkConfigurationDetails': None,
  'logGroupId': None,
  'aicUserSchemaName': None}}

In [2]:
# ---------------------------------------------------------------------------
# Test 05 — Create ADW external catalog
# ---------------------------------------------------------------------------

def test_05():
    """POST /catalogs  — create ADW external catalog in the AIDP instance."""
    print(f"\n=== test_05: Create ADW catalog '{config.CATALOG_NAME}' ===")

    assert config.WORKSPACE_KEY, "Set WORKSPACE_KEY in config.py (from test_02 or test_03)"
    assert config.ADW_PASSWORD, "Set ADW_PASSWORD in config.py"
    assert config.ADW_TNS_ALIAS, "Set ADW_TNS_ALIAS in config.py"

    wallet_b64 = ""
    if config.ADW_WALLET_ZIP:
        if not os.path.exists(config.ADW_WALLET_ZIP):
            print(f"  WARNING: Wallet file not found: {config.ADW_WALLET_ZIP} — sending empty wallet")
        else:
            with open(config.ADW_WALLET_ZIP, "rb") as f:
                wallet_b64 = base64.b64encode(f.read()).decode("utf-8")
            print(f"  Wallet: {config.ADW_WALLET_ZIP} ({len(wallet_b64)} bytes base64)")

    client = make_client()
    body = {
        "displayName": config.CATALOG_NAME,
        "description": f"Test ADW catalog: {config.CATALOG_NAME}",
        "catalogType": "EXTERNAL",
        "sourceType": "ADW",
        "connectionDetails": {
            "connectionProperties": {
                "ADW_WALLET_CONTENT_BASE64": wallet_b64,
                "ADW_WALLET_PASSWORD": config.ADW_WALLET_PASSWORD or "",
                "ADW_USERNAME": config.ADW_USERNAME,
                "ADW_PASSWORD": config.ADW_PASSWORD,
                "ADW_TNS_ALIAS": config.ADW_TNS_ALIAS,
            },
        },
    }

    # Print body without secrets
    safe_body = json.loads(json.dumps(body))
    safe_body["connectionDetails"]["connectionProperties"]["ADW_PASSWORD"] = "***"
    safe_body["connectionDetails"]["connectionProperties"]["ADW_WALLET_CONTENT_BASE64"] = \
        f"<{len(wallet_b64)} chars>" if wallet_b64 else ""
    print(f"  Request body (secrets redacted):")
    print(json.dumps(safe_body, indent=2))

    resp = client.post("/catalogs", body)
    print_response(resp, "POST /catalogs")

    assert resp.status_code in (200, 201, 202), \
        f"Expected 200/201/202, got {resp.status_code}: {resp.text[:300]}"

    data = resp.json() if resp.content else {}
    cat_key = data.get("key") or data.get("id")
    print(f"  Catalog key: {cat_key}")
    print("  PASS")
    return {"key": cat_key, "data": data}


In [4]:
test_05()


=== test_05: Create ADW catalog 'PilotADWDB' ===
  Wallet: ./Wallet_pilot.zip (33784 bytes base64)
  Request body (secrets redacted):
{
  "displayName": "PilotADWDB",
  "description": "Test ADW catalog: PilotADWDB",
  "catalogType": "EXTERNAL",
  "sourceType": "ADW",
  "connectionDetails": {
    "connectionProperties": {
      "ADW_WALLET_CONTENT_BASE64": "<33784 chars>",
      "ADW_WALLET_PASSWORD": "Welcome#1234!",
      "ADW_USERNAME": "ADMIN",
      "ADW_PASSWORD": "***",
      "ADW_TNS_ALIAS": "pilot_low"
    }
  }
}
  HTTP 202 [POST /catalogs]

  Catalog key: None
  PASS


{'key': None, 'data': {}}

In [5]:
# ---------------------------------------------------------------------------
# Test 06 — Create Spark compute cluster
# ---------------------------------------------------------------------------

def test_06():
    """POST /workspaces/{ws_key}/clusters  — create Spark cluster."""
    print(f"\n=== test_06: Create cluster '{config.CLUSTER_NAME}' ===")

    assert config.WORKSPACE_KEY, "Set WORKSPACE_KEY in config.py (from test_02 or test_03)"

    client = make_client()
    body = {
        "displayName": config.CLUSTER_NAME,
        "description": f"Test cluster: {config.CLUSTER_NAME}",
        "clusterRuntimeConfig": {
            "type": "SPARK",
            "sparkVersion": "3.5.0",
        },
        "driverConfig": {
            "driverShape": "amd.generic",
            "driverShapeConfig": {
                "ocpus": config.DRIVER_OCPUS,
                "gpus": 0,
                "memoryInGBs": config.DRIVER_MEMORY_GBS,
            },
        },
        "workerConfig": {
            "workerShape": "amd.generic",
            "workerShapeConfig": {
                "ocpus": config.WORKER_OCPUS,
                "gpus": 0,
                "memoryInGBs": config.WORKER_MEMORY_GBS,
            },
            "minWorkerCount": config.MIN_WORKERS,
            "maxWorkerCount": config.MAX_WORKERS,
        },
    }

    print(f"  Request body:")
    print(json.dumps(body, indent=2))

    resp = client.post(f"/workspaces/{config.WORKSPACE_KEY}/clusters", body)
    print_response(resp, "POST /clusters")

    assert resp.status_code in (200, 201, 202), \
        f"Expected 200/201/202, got {resp.status_code}: {resp.text[:300]}"

    data = resp.json() if resp.content else {}
    cluster_key = data.get("key") or data.get("id")

    if not cluster_key:
        print("  No key in response — polling list ...")
        import time
        end = time.time() + 120
        while time.time() < end:
            time.sleep(10)
            list_resp = client.get(f"/workspaces/{config.WORKSPACE_KEY}/clusters")
            if list_resp.ok:
                items = list_resp.json()
                if isinstance(items, dict):
                    items = items.get("items", [])
                for item in items:
                    if item.get("displayName") == config.CLUSTER_NAME:
                        cluster_key = item.get("key") or item.get("id")
                        if cluster_key:
                            break
                if cluster_key:
                    break

    assert cluster_key, "Could not retrieve cluster key after creation"

    cluster_data = poll_until_active(
        client,
        f"/workspaces/{config.WORKSPACE_KEY}/clusters/{cluster_key}",
        f"cluster '{config.CLUSTER_NAME}'",
        poll_interval=20, max_wait=900,
    )

    print(f"\n  >>> Copy this to config.py CLUSTER_KEY: {cluster_key}")
    print("  PASS")
    return {"key": cluster_key, "data": cluster_data}


In [6]:
test_06()


=== test_06: Create cluster 'Pilot_api_Cluster1' ===
  Request body:
{
  "displayName": "Pilot_api_Cluster1",
  "description": "Test cluster: Pilot_api_Cluster1",
  "clusterRuntimeConfig": {
    "type": "SPARK",
    "sparkVersion": "3.5.0"
  },
  "driverConfig": {
    "driverShape": "amd.generic",
    "driverShapeConfig": {
      "ocpus": 2,
      "gpus": 0,
      "memoryInGBs": 32
    }
  },
  "workerConfig": {
    "workerShape": "amd.generic",
    "workerShapeConfig": {
      "ocpus": 2,
      "gpus": 0,
      "memoryInGBs": 32
    },
    "minWorkerCount": 1,
    "maxWorkerCount": 2
  }
}
  HTTP 202 [POST /clusters]
{
  "sourceApi": "CLUSTER_API",
  "key": "7696453b-2df7-4c89-a64f-ac90d668a357",
  "displayName": "Pilot_api_Cluster1",
  "description": "Test cluster: Pilot_api_Cluster1",
  "type": "USER",
  "timeCreated": "2026-04-21T13:17:27.292Z",
  "timeUpdated": "2026-04-21T13:17:27.292Z",
  "state": "ACCEPTED",
  "stateDetails": null,
  "nodeType": null,
  "driverConfig": {
    "

{'key': '7696453b-2df7-4c89-a64f-ac90d668a357',
 'data': {'sourceApi': 'CLUSTER_API',
  'key': '7696453b-2df7-4c89-a64f-ac90d668a357',
  'displayName': 'Pilot_api_Cluster1',
  'description': 'Test cluster: Pilot_api_Cluster1',
  'type': 'USER',
  'timeCreated': '2026-04-21T13:17:27.292Z',
  'timeUpdated': '2026-04-21T13:23:47.615Z',
  'state': 'ACTIVE',
  'stateDetails': None,
  'nodeType': None,
  'driverConfig': {'driverNodeType': None,
   'driverShape': 'amd.generic',
   'driverShapeConfig': {'ocpus': 2, 'gpus': 0, 'memoryInGBs': 32}},
  'activeClusterResources': {'activeExecutorCount': 1.0,
   'activeCores': 8.0,
   'activeGpuCores': 0.0,
   'activeMemoryInGBs': 64.0,
   'activeGpuMemoryInGBs': 0.0},
  'createdBy': 'ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta',
  'createdByName': 'rajib.g.ghosh@oracle.com',
  'updatedBy': None,
  'updatedByName': None,
  'stoppedBy': None,
  'stoppedByName': None,
  'freeformTags': None,
  'definedTags': None,
  'wo

In [2]:
# ---------------------------------------------------------------------------
# Test 07 — Get cluster connection details
# ---------------------------------------------------------------------------

def test_07():
    """GET /workspaces/{ws_key}/clusters/{cluster_key}  — fetch JDBC/Thrift details."""
    print(f"\n=== test_07: Get cluster connection details ===")

    assert config.WORKSPACE_KEY, "Set WORKSPACE_KEY in config.py"
    assert config.CLUSTER_KEY, "Set CLUSTER_KEY in config.py (from test_06)"

    client = make_client()
    resp = client.get(f"/workspaces/{config.WORKSPACE_KEY}/clusters/{config.CLUSTER_KEY}")
    print_response(resp, "GET cluster")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    data = resp.json()

    connection_fields = ["jdbcUrl", "jdbcConnectionUrl", "thriftServerUrl",
                         "connectionDetails", "sparkJdbcUrl", "beeline",
                         "hostName", "host", "port", "httpPath",
                         "connectionProperties", "clusterConnectionDetails"]
    found = {k: data[k] for k in connection_fields if k in data}
    if found:
        print(f"  Connection fields found:")
        print(json.dumps(found, indent=2, default=str))
    else:
        print("  No JDBC/Thrift fields found in response — full (safe) response above")

    print(f"  Spark version: {data.get('clusterRuntimeConfig', {}).get('sparkVersion')}")
    print(f"  State        : {data.get('lifecycleState')}")
    print("  PASS")
    return data



In [3]:
test_07()


=== test_07: Get cluster connection details ===
  HTTP 200 [GET cluster]
{
  "sourceApi": "CLUSTER_API",
  "key": "7696453b-2df7-4c89-a64f-ac90d668a357",
  "displayName": "Pilot_api_Cluster1",
  "description": "Test cluster: Pilot_api_Cluster1",
  "type": "USER",
  "timeCreated": "2026-04-21T13:17:27.292Z",
  "timeUpdated": "2026-04-21T13:24:31.095Z",
  "state": "ACTIVE",
  "stateDetails": null,
  "nodeType": null,
  "driverConfig": {
    "driverNodeType": null,
    "driverShape": "amd.generic",
    "driverShapeConfig": {
      "ocpus": 2,
      "gpus": 0,
      "memoryInGBs": 32
    }
  },
  "activeClusterResources": {
    "activeExecutorCount": 1.0,
    "activeCores": 8.0,
    "activeGpuCores": 0.0,
    "activeMemoryInGBs": 64.0,
    "activeGpuMemoryInGBs": 0.0
  },
  "createdBy": "ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta",
  "createdByName": "rajib.g.ghosh@oracle.com",
  "updatedBy": null,
  "updatedByName": null,
  "stoppedBy": null,
  "stoppedB

{'sourceApi': 'CLUSTER_API',
 'key': '7696453b-2df7-4c89-a64f-ac90d668a357',
 'displayName': 'Pilot_api_Cluster1',
 'description': 'Test cluster: Pilot_api_Cluster1',
 'type': 'USER',
 'timeCreated': '2026-04-21T13:17:27.292Z',
 'timeUpdated': '2026-04-21T13:24:31.095Z',
 'state': 'ACTIVE',
 'stateDetails': None,
 'nodeType': None,
 'driverConfig': {'driverNodeType': None,
  'driverShape': 'amd.generic',
  'driverShapeConfig': {'ocpus': 2, 'gpus': 0, 'memoryInGBs': 32}},
 'activeClusterResources': {'activeExecutorCount': 1.0,
  'activeCores': 8.0,
  'activeGpuCores': 0.0,
  'activeMemoryInGBs': 64.0,
  'activeGpuMemoryInGBs': 0.0},
 'createdBy': 'ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta',
 'createdByName': 'rajib.g.ghosh@oracle.com',
 'updatedBy': None,
 'updatedByName': None,
 'stoppedBy': None,
 'stoppedByName': None,
 'freeformTags': None,
 'definedTags': None,
 'workerConfig': {'workerShape': 'amd.generic',
  'workerShapeConfig': {'ocpus': 2, 'gp

In [4]:
# ---------------------------------------------------------------------------
# Test 08 — List clusters in workspace
# ---------------------------------------------------------------------------

def test_08():
    """GET /workspaces/{ws_key}/clusters  — list all clusters."""
    print(f"\n=== test_08: List clusters in workspace ===")

    assert config.WORKSPACE_KEY, "Set WORKSPACE_KEY in config.py"

    client = make_client()
    resp = client.get(f"/workspaces/{config.WORKSPACE_KEY}/clusters")
    print_response(resp, "GET /clusters")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    items = resp.json()
    if isinstance(items, dict):
        items = items.get("items", [])
    print(f"  Found {len(items)} cluster(s):")
    for c in items:
        print(f"    key={c.get('key')}  name={c.get('displayName')}  state={c.get('lifecycleState')}")
    print("  PASS")
    return items




In [5]:
test_08()


=== test_08: List clusters in workspace ===
  HTTP 200 [GET /clusters]
{
  "items": [
    {
      "key": "7696453b-2df7-4c89-a64f-ac90d668a357",
      "displayName": "Pilot_api_Cluster1",
      "description": "Test cluster: Pilot_api_Cluster1",
      "type": "USER",
      "timeCreated": "2026-04-21T13:17:27.292Z",
      "timeUpdated": "2026-04-21T13:24:31.095Z",
      "state": "ACTIVE",
      "stateDetails": null,
      "createdBy": "ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta",
      "createdByName": "rajib.g.ghosh@oracle.com",
      "updatedBy": null,
      "updatedByName": null,
      "stoppedBy": null,
      "stoppedByName": null,
      "clusterRuntimeConfig": {
        "type": "SPARK",
        "initScripts": null,
        "sparkVersion": "3.5.0",
        "sparkAdvancedConfigurations": null,
        "sparkEnvVariables": null
      },
      "activeClusterResources": {
        "activeExecutorCount": 1.0,
        "activeCores": 8.0,
        "activeGpu

[{'key': '7696453b-2df7-4c89-a64f-ac90d668a357',
  'displayName': 'Pilot_api_Cluster1',
  'description': 'Test cluster: Pilot_api_Cluster1',
  'type': 'USER',
  'timeCreated': '2026-04-21T13:17:27.292Z',
  'timeUpdated': '2026-04-21T13:24:31.095Z',
  'state': 'ACTIVE',
  'stateDetails': None,
  'createdBy': 'ocid1.user.oc1..aaaaaaaay7jgelqticzmtchjk7rh2pgeo7a4eusiguyr2go3rg22ju523yta',
  'createdByName': 'rajib.g.ghosh@oracle.com',
  'updatedBy': None,
  'updatedByName': None,
  'stoppedBy': None,
  'stoppedByName': None,
  'clusterRuntimeConfig': {'type': 'SPARK',
   'initScripts': None,
   'sparkVersion': '3.5.0',
   'sparkAdvancedConfigurations': None,
   'sparkEnvVariables': None},
  'activeClusterResources': {'activeExecutorCount': 1.0,
   'activeCores': 8.0,
   'activeGpuCores': 0.0,
   'activeMemoryInGBs': 64.0,
   'activeGpuMemoryInGBs': 0.0},
  'driverConfig': {'driverNodeType': None,
   'driverShape': 'amd.generic',
   'driverShapeConfig': {'ocpus': 2, 'gpus': 0, 'memoryInGBs

In [6]:
# ---------------------------------------------------------------------------
# Test 09 — List catalogs in AIDP instance
# ---------------------------------------------------------------------------

def test_09():
    """GET /catalogs  — list all catalogs."""
    print(f"\n=== test_09: List catalogs ===")

    client = make_client()
    resp = client.get("/catalogs")
    print_response(resp, "GET /catalogs")

    assert resp.ok, f"Expected 2xx, got {resp.status_code}"
    items = resp.json()
    if isinstance(items, dict):
        items = items.get("items", [])
    print(f"  Found {len(items)} catalog(s):")
    for c in items:
        print(f"    key={c.get('key')}  name={c.get('displayName')}  state={c.get('lifecycleState')}")
    print("  PASS")
    return items


In [7]:
test_09()


=== test_09: List catalogs ===
  HTTP 200 [GET /catalogs]
{
  "items": [
    {
      "key": "dev_lake_sql",
      "displayName": "dev_lake_sql",
      "description": null,
      "catalogType": "INTERNAL",
      "catalogGuid": "3f32c4a8d71446af99df72d475a77ba4",
      "sourceType": null,
      "lifecycleState": "ACTIVE",
      "lifecycleStateDetails": null,
      "timeCreated": "2025-10-18T14:25:45.950Z",
      "timeUpdated": null,
      "createdBy": "rajib.g.ghosh@oracle.com",
      "updatedBy": null,
      "lastRefreshStatus": null,
      "timeLastRefresh": null
    },
    {
      "key": "adw_gold",
      "displayName": "adw_gold",
      "description": null,
      "catalogType": "EXTERNAL",
      "catalogGuid": "60fa15169f524a448ffafcaf9aadea87",
      "sourceType": "ADW",
      "lifecycleState": "ACTIVE",
      "lifecycleStateDetails": null,
      "timeCreated": "2025-11-20T19:07:43.872Z",
      "timeUpdated": null,
      "createdBy": "rajib.g.ghosh@oracle.com",
      "updatedBy": n

[{'key': 'dev_lake_sql',
  'displayName': 'dev_lake_sql',
  'description': None,
  'catalogType': 'INTERNAL',
  'catalogGuid': '3f32c4a8d71446af99df72d475a77ba4',
  'sourceType': None,
  'lifecycleState': 'ACTIVE',
  'lifecycleStateDetails': None,
  'timeCreated': '2025-10-18T14:25:45.950Z',
  'timeUpdated': None,
  'createdBy': 'rajib.g.ghosh@oracle.com',
  'updatedBy': None,
  'lastRefreshStatus': None,
  'timeLastRefresh': None},
 {'key': 'adw_gold',
  'displayName': 'adw_gold',
  'description': None,
  'catalogType': 'EXTERNAL',
  'catalogGuid': '60fa15169f524a448ffafcaf9aadea87',
  'sourceType': 'ADW',
  'lifecycleState': 'ACTIVE',
  'lifecycleStateDetails': None,
  'timeCreated': '2025-11-20T19:07:43.872Z',
  'timeUpdated': None,
  'createdBy': 'rajib.g.ghosh@oracle.com',
  'updatedBy': None,
  'lastRefreshStatus': 'SUCCESS',
  'timeLastRefresh': None},
 {'key': 'proxytest',
  'displayName': 'proxytest',
  'description': None,
  'catalogType': 'EXTERNAL',
  'catalogGuid': '9b6443

# ---------------------------------------------------------------------------
# Runner
# ---------------------------------------------------------------------------

ALL_TESTS = {
    "test_01": test_01,
    "test_02": test_02,
    "test_03": test_03,
    "test_04": test_04,
    "test_05": test_05,
    "test_06": test_06,
    "test_07": test_07,
    "test_08": test_08,
    "test_09": test_09,
}

# Tests safe to run without WORKSPACE_KEY / CLUSTER_KEY set
READ_ONLY_TESTS = ["test_01", "test_02", "test_03"]


if __name__ == "__main__":
    if len(sys.argv) > 1:
        # Run a specific test: python3 tests.py test_03
        name = sys.argv[1]
        if name not in ALL_TESTS:
            print(f"Unknown test '{name}'. Available: {', '.join(ALL_TESTS)}")
            sys.exit(1)
        ALL_TESTS[name]()
    else:
        # Run read-only tests by default (no side effects)
        print("Running read-only tests (01, 02, 03).")
        print("For mutating tests (04–09), run: python3 tests.py test_04")
        for name in READ_ONLY_TESTS:
            ALL_TESTS[name]()
        print("\nAll read-only tests passed.")

## 